In [0]:
from pyspark.sql.functions import sum, avg, round

# monthly sales trend
monthly_sales = spark.read.table("ecommerce.e_comm_gold.factSales")
monthly_sales.createOrReplaceTempView("customer_revenue_summary")

# calculate metrics 
agg_monthly_sales = spark.sql("""        
        select 
            customer_key
            , count(order_id) as total_orders
            , sum(order_quantity) as total_items_purchased
            , round(sum(sale_amount),2) as total_spent
            , round(avg(sale_amount),2) as avg_order_value
            , round(avg(order_quantity),2) as avg_items_purchased_per_order
            , current_timestamp() as load_ts
        from customer_revenue_summary
        where dq_note = "is_valid"
        group by customer_key
        """)
    
# write to delta table

agg_monthly_sales.write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("ecommerce.e_comm_gold.fact_agg_customer_revenue_summary")